In [3]:
import pandas as pd
import numpy as np
import re
from typing import List, Dict
print("Pandas is working")

Pandas is working


### Importing Libraries

In [4]:
!pip install pandas nltk
import pandas as pd
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Deepalakshmi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Deepalakshmi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

### Load the dataset

In [5]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Deepalakshmi\Downloads\sample_emails_with_triage_200.csv")


### Displaying records

In [6]:
df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,notify,urgent
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,polite
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,notify,urgent
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,polite
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,polite


### Cleaning email text

In [7]:
df['clean_text'] = (
    df['body']
    .str.lower()
    .str.replace('[^a-zA-Z ]', '', regex=True)
)

df[['body', 'clean_text']].head()

,body,clean_text
0,Reminder: The client meeting is scheduled at 1...,reminder the client meeting is scheduled at t...
1,Your invoice of INR 25515.09 is due on 2025-12...,your invoice of inr is due on please pay to ...
2,Reminder: The client meeting is scheduled at 1...,reminder the client meeting is scheduled at t...
3,"Hello team, please find the attached weekly re...",hello team please find the attached weekly rep...
4,"Hello team, please find the attached weekly re...",hello team please find the attached weekly rep...


### Keyword Extraction

In [8]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words("english"))

df['keywords'] = df['clean_text'].apply(
    lambda x: [w for w in x.split() if w not in stop_words]
)

df[['clean_text', 'keywords']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Deepalakshmi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,clean_text,keywords
0,reminder the client meeting is scheduled at t...,"[reminder, client, meeting, scheduled, tomorro..."
1,your invoice of inr is due on please pay to ...,"[invoice, inr, due, please, pay, avoid, late, ..."
2,reminder the client meeting is scheduled at t...,"[reminder, client, meeting, scheduled, tomorro..."
3,hello team please find the attached weekly rep...,"[hello, team, please, find, attached, weekly, ..."
4,hello team please find the attached weekly rep...,"[hello, team, please, find, attached, weekly, ..."


In [12]:
df['predicted_triage'] = df['clean_text'].apply(triage_rule)
df[['body','predicted_triage']].head()

,body,predicted_triage
0,Reminder: The client meeting is scheduled at 1...,respond_or_act
1,Your invoice of INR 25515.09 is due on 2025-12...,respond_or_act
2,Reminder: The client meeting is scheduled at 1...,respond_or_act
3,"Hello team, please find the attached weekly re...",respond_or_act
4,"Hello team, please find the attached weekly re...",respond_or_act


###  Rule-based triage function 

In [11]:
def triage_rule(text):
    text = text.lower()

    if "refund" in text or "urgent" in text or "password" in text:
        return "notify_human"

    if "newsletter" in text or "promotion" in text:
        return "ignore"

    return "respond_or_act" 

df['triage'] = df['clean_text'].apply(triage_rule)
df[['body', 'triage']].head()

,body,triage
0,Reminder: The client meeting is scheduled at 1...,respond_or_act
1,Your invoice of INR 25515.09 is due on 2025-12...,respond_or_act
2,Reminder: The client meeting is scheduled at 1...,respond_or_act
3,"Hello team, please find the attached weekly re...",respond_or_act
4,"Hello team, please find the attached weekly re...",respond_or_act


In [13]:
df.to_csv(r"C:\Users\Deepalakshmi\Downloads\data\milestone1_output_lingaeswari.csv", index=False)

In [14]:
import pandas as pd
import re
from IPython.display import display

# adjust path if needed
CSV_PATH = r"C:\Users\Deepalakshmi\Downloads/sample_emails_with_triage_200.csv"
df = pd.read_csv(CSV_PATH)
print("Loaded rows:", len(df))
display(df.head())

Loaded rows: 200


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,notify,urgent
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,polite
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,notify,urgent
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,polite
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,polite


In [15]:
def make_clean_text(s):
    if not isinstance(s, str):
        return ""
    s = s.strip()
    # normalize spaces and lowercase
    s = re.sub(r'\s+', ' ', s)
    s = s.lower()
    return s

# prefer 'body' column; if not, try 'email_body' or 'subject'
source_col = 'body' if 'body' in df.columns else ('email_body' if 'email_body' in df.columns else 'subject')
df['clean_text'] = df[source_col].astype(str).apply(make_clean_text)
display(df[[source_col,'clean_text']].head())

,body,clean_text
0,Reminder: The client meeting is scheduled at 1...,reminder: the client meeting is scheduled at 1...
1,Your invoice of INR 25515.09 is due on 2025-12...,your invoice of inr 25515.09 is due on 2025-12...
2,Reminder: The client meeting is scheduled at 1...,reminder: the client meeting is scheduled at 1...
3,"Hello team, please find the attached weekly re...","hello team, please find the attached weekly re..."
4,"Hello team, please find the attached weekly re...","hello team, please find the attached weekly re..."


###  Email triage system for more precise pattern matching

In [16]:
RULES = {
    'notify_human': [
        r'\binvoice\b', r'\bbill\b', r'\bpayment\b', r'\bpassword\b', r'\breset\b',
        r'\bfraud\b', r'\bchargeback\b', r'\bdispute\b', r'\bdenied\b', r'\boverdue\b'
    ],
    'respond_or_act': [
        r'\bmeeting\b', r'\bschedule\b', r'\bscheduled\b', r'\battached\b', r'\battachment\b',
        r'\bplease find\b', r'\bplease review\b', r'\baction required\b', r'\burgent\b', r'\basap\b'
    ],
    'ignore': [
        r'\bunsubscribe\b', r'\bpromotion\b', r'\bsale\b', r'\boffer\b', r'\bnewsletter\b',
        r'\bsurvey\b', r'\bcongratulations\b'
    ]
}
COMPILED = {k:[re.compile(p, flags=re.I) for p in v] for k,v in RULES.items()}

def triage_rule_debug(text):
    if not isinstance(text, str) or text.strip()=="":
        return ('ignore','empty_text')
    for label in ['notify_human','respond_or_act','ignore']:
        for pat in COMPILED[label]:
            if pat.search(text):
                return (label, pat.pattern)
    return ('respond_or_act','default_fallback')

# apply and store both label and matched pattern
df[['triage','triage_match']] = df['clean_text'].apply(lambda t: pd.Series(triage_rule_debug(t)))
print("Triage value counts (advanced):")
print(df['triage'].value_counts())
display(df[['clean_text','triage','triage_match']].head(20))

Triage value counts (advanced):
triage
respond_or_act    127
ignore             42
notify_human       31
Name: count, dtype: int64


,clean_text,triage,triage_match
0,reminder: the client meeting is scheduled at 1...,respond_or_act,\bmeeting\b
1,your invoice of inr 25515.09 is due on 2025-12...,notify_human,\binvoice\b
2,reminder: the client meeting is scheduled at 1...,respond_or_act,\bmeeting\b
3,"hello team, please find the attached weekly re...",respond_or_act,\battached\b
4,"hello team, please find the attached weekly re...",respond_or_act,\battached\b
5,your invoice of inr 38766.88 is due on 2025-12...,notify_human,\binvoice\b
6,congratulations! you have been selected as a l...,ignore,\bcongratulations\b
7,your order #6313 has been shipped and is expec...,respond_or_act,default_fallback
8,"dear user, we detected a login from a new devi...",notify_human,\bpassword\b
9,your invoice of inr 32944.21 is due on 2025-12...,notify_human,\binvoice\b


### Displaying specific subsets of the DataFrame

In [17]:
# 5. Diagnostics — use this in live demo to show  why each label was chosen
print("Top 20 rows with triage reasons:")
display(df[['body','clean_text','triage','triage_match']].head(20))

# show a few sample rows where triage==notify_human for inspection
print("\nSample notify_human rows:")
display(df[df['triage']=='notify_human'][['body','clean_text','triage_match']].head(6))

print("\nSample respond_or_act rows:")
display(df[df['triage']=='respond_or_act'][['body','clean_text','triage_match']].head(6))

print("\nSample ignore rows:")
display(df[df['triage']=='ignore'][['body','clean_text','triage_match']].head(6))

Top 20 rows with triage reasons:


,body,clean_text,triage,triage_match
0,Reminder: The client meeting is scheduled at 1...,reminder: the client meeting is scheduled at 1...,respond_or_act,\bmeeting\b
1,Your invoice of INR 25515.09 is due on 2025-12...,your invoice of inr 25515.09 is due on 2025-12...,notify_human,\binvoice\b
2,Reminder: The client meeting is scheduled at 1...,reminder: the client meeting is scheduled at 1...,respond_or_act,\bmeeting\b
3,"Hello team, please find the attached weekly re...","hello team, please find the attached weekly re...",respond_or_act,\battached\b
4,"Hello team, please find the attached weekly re...","hello team, please find the attached weekly re...",respond_or_act,\battached\b
5,Your invoice of INR 38766.88 is due on 2025-12...,your invoice of inr 38766.88 is due on 2025-12...,notify_human,\binvoice\b
6,Congratulations! You have been selected as a l...,congratulations! you have been selected as a l...,ignore,\bcongratulations\b
7,Your order #6313 has been shipped and is expec...,your order #6313 has been shipped and is expec...,respond_or_act,default_fallback
8,"Dear user, we detected a login from a new devi...","dear user, we detected a login from a new devi...",notify_human,\bpassword\b
9,Your invoice of INR 32944.21 is due on 2025-12...,your invoice of inr 32944.21 is due on 2025-12...,notify_human,\binvoice\b



Sample notify_human rows:


,body,clean_text,triage_match
1,Your invoice of INR 25515.09 is due on 2025-12...,your invoice of inr 25515.09 is due on 2025-12...,\binvoice\b
5,Your invoice of INR 38766.88 is due on 2025-12...,your invoice of inr 38766.88 is due on 2025-12...,\binvoice\b
8,"Dear user, we detected a login from a new devi...","dear user, we detected a login from a new devi...",\bpassword\b
9,Your invoice of INR 32944.21 is due on 2025-12...,your invoice of inr 32944.21 is due on 2025-12...,\binvoice\b
24,Your invoice of INR 9983.71 is due on 2025-12-...,your invoice of inr 9983.71 is due on 2025-12-...,\binvoice\b
27,"Dear user, we detected a login from a new devi...","dear user, we detected a login from a new devi...",\bpassword\b



Sample respond_or_act rows:


,body,clean_text,triage_match
0,Reminder: The client meeting is scheduled at 1...,reminder: the client meeting is scheduled at 1...,\bmeeting\b
2,Reminder: The client meeting is scheduled at 1...,reminder: the client meeting is scheduled at 1...,\bmeeting\b
3,"Hello team, please find the attached weekly re...","hello team, please find the attached weekly re...",\battached\b
4,"Hello team, please find the attached weekly re...","hello team, please find the attached weekly re...",\battached\b
7,Your order #6313 has been shipped and is expec...,your order #6313 has been shipped and is expec...,default_fallback
10,Reminder: The client meeting is scheduled at 1...,reminder: the client meeting is scheduled at 1...,\bmeeting\b



Sample ignore rows:


,body,clean_text,triage_match
6,Congratulations! You have been selected as a l...,congratulations! you have been selected as a l...,\bcongratulations\b
15,"Hi, don't miss our sale with discounts up to 7...","hi, don't miss our sale with discounts up to 7...",\bsale\b
17,"Hi, don't miss our sale with discounts up to 7...","hi, don't miss our sale with discounts up to 7...",\bsale\b
28,"Hi, don't miss our sale with discounts up to 7...","hi, don't miss our sale with discounts up to 7...",\bsale\b
32,Congratulations! You have been selected as a l...,congratulations! you have been selected as a l...,\bcongratulations\b
34,Congratulations! You have been selected as a l...,congratulations! you have been selected as a l...,\bcongratulations\b


### Saving the output to csv file

In [18]:
# 6. Save results for submission
OUT = r"C:/Users/Deepalakshmi/Downloads/data/milestone1_output_lingaeswari.csv"
df.to_csv(OUT, index=False)


In [19]:
def read_calender() ->str:
    return "No meetings scheduled."
def send_auto_reply() -> str:
    return "Auto-reply sent."

In [20]:
def react_agent(email: Dict) -> Dict:
    thought = "Classify email intent using triage rules."

    action = triage_email(email["subject"],email["body"])

    if action == "respond":
        observation = send_auto_reply()
    elif action == "notify_human":
        observation = "Human notified."
    else:
        observation = "Email ignored."
    return {
        "thought":thought,
        "action": action,
        "observation": observation
    }

In [21]:
from typing import Dict

def react_agent(email: Dict) -> Dict:
    """
    Processes an email record using specific data columns.
    Expected keys: 'subject', 'body', 'sender', 'priority', etc.
    """
    # 1. Thought: Use 'clean_text' or 'subject' for context
    # Safely get values to prevent KeyError if a column is missing
    subject = email.get('subject', 'No Subject')
    priority = email.get('priority', 'normal')
    
    thought = f"Processing email ID {email.get('id')} from {email.get('sender')}. Priority: {priority}."

    # 2. Action: Use triage-specific columns to determine the path
    # We pass 'subject' and 'body' as originally intended
    action = triage_email(email.get('subject', ''), email.get('body', ''))

    # 3. Observation: Logic based on the 'action' result
    if action == "respond":
        observation = f"Auto-reply sent. Triage Match Status: {email.get('triage_match')}"
    elif action == "notify_human":
        observation = f"Human notified. Triage Label: {email.get('triage_label')}"
    else:
        observation = "Email ignored based on triage logic."

    return {
        "thought": thought,
        "action": action,
        "observation": observation
    }

def triage_email(subject: str, body: str) -> str:
    # Logic using your column names
    if "urgent" in subject.lower():
        return "notify_human"
    return "respond"

In [22]:
results: List[Dict] = []
for _, row in df.iterrows():
    agent_result = react_agent(row)
    results.append({
        "id" :row["id"],
        "triage_label" : row["triage_label"],
        "predicted_triage": agent_result["action"]
        })
df_results = pd.DataFrame(results)
df_results.head()

,id,triage_label,predicted_triage
0,1,notify_human,respond
1,2,respond,respond
2,3,ignore,respond
3,4,respond,respond
4,5,respond,respond


In [23]:
df['predicted_triage'] = df['clean_text'].apply(triage_rule)
df[['body','predicted_triage']].head()

,body,predicted_triage
0,Reminder: The client meeting is scheduled at 1...,respond_or_act
1,Your invoice of INR 25515.09 is due on 2025-12...,respond_or_act
2,Reminder: The client meeting is scheduled at 1...,respond_or_act
3,"Hello team, please find the attached weekly re...",respond_or_act
4,"Hello team, please find the attached weekly re...",respond_or_act


In [24]:
df[['body','predicted_triage']].to_csv("C:/Users/Deepalakshmi/Downloads/data/milestone1_output_lingaeswari.csv")